## Phần 1: Cài đặt và Kiểm chứng OLS, Hat Matrix, Ridge Regression và VIF
Notebook này trình bày công thức toán học, cài đặt từ scratch và kiểm chứng bốn hàm cốt lõi so với NumPy/sklearn.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge as SklearnRidge
from sklearn.linear_model import LinearRegression

from ols_implementation import ols_fit, hat_matrix, vif
from ridge_lasso import ridge_fit
from test_synthetic_data import generate_synthetic_data

# Tao du lieu gia lap co gia tri beta thuc va nhieu Gaussian
np.random.seed(42)
n, p = 100, 3
X_toy = np.random.randn(n, p)
X_design = np.c_[np.ones(n), X_toy]   # them cot he so chan
true_beta = np.array([5.0, 2.0, -3.0, 1.5])
y_toy = X_design @ true_beta + np.random.randn(n) * 1.5

print('X_design shape:', X_design.shape)
print('y shape       :', y_toy.shape)
print('true_beta     :', true_beta)

### 1. OLS Fit — Phương pháp Bình phương Bé nhất Thông thường

**Bài toán tối ưu:**  
Tìm $\hat{\beta}$ sao cho $\| y - X\beta \|^2$ đạt cực tiểu.

**Dẫn xuất công thức (Normal Equation):**
$$\frac{\partial}{\partial \beta} \| y - X\beta \|^2 = -2 X^T (y - X\beta) = 0 \implies X^T X \hat{\beta} = X^T y$$

Khi $X^T X$ khả nghịch, nghiệm duy nhất là:
$$\hat{\beta} = (X^T X)^{-1} X^T y$$

**Cài đặt — Economic SVD ($X = U \Sigma V^T$):**
$$\hat{\beta} = V \Sigma^{-1} U^T y$$
Ổn định hơn nghịch đảo trực tiếp khi $X^T X$ gần kỳ dị.

* **Kiểm chứng:** So sánh $\hat{\beta}_{\text{custom}}$ với `sklearn.LinearRegression`.

In [ ]:
beta_custom = ols_fit(X_design, y_toy)

# Kiem chung voi sklearn
lr = LinearRegression(fit_intercept=False)
lr.fit(X_design, y_toy)
beta_sklearn = lr.coef_

# Kiem chung voi numpy lstsq
beta_lstsq = np.linalg.lstsq(X_design, y_toy, rcond=None)[0]

print(f'{'':20s} {'beta_0':>10s} {'beta_1':>10s} {'beta_2':>10s} {'beta_3':>10s}')
print(f'{'true_beta':20s}', ' '.join(f'{v:10.6f}' for v in true_beta))
print(f'{'ols_fit (custom)':20s}', ' '.join(f'{v:10.6f}' for v in beta_custom))
print(f'{'sklearn':20s}', ' '.join(f'{v:10.6f}' for v in beta_sklearn))
print()
print(f'Max |custom - sklearn| : {np.max(np.abs(beta_custom - beta_sklearn)):.2e}')
print(f'Max |custom - lstsq|   : {np.max(np.abs(beta_custom - beta_lstsq)):.2e}')

* **Nhận xét:** $\hat{\beta}_{\text{custom}}$ khớp với `sklearn` và `numpy.linalg.lstsq` với sai số $\leq 10^{-14}$ (giới hạn dấu phẩy động IEEE 754). Hàm `ols_fit` hoạt động chính xác.

### 2. Hat Matrix — Ma trận Chiếu

Ma trận $H$ chiếu $y$ lên không gian cột của $X$:
$$H = X(X^T X)^{-1} X^T, \quad \hat{y} = Hy$$

**Tính chất quan trọng:**
* **Lũy đẳng (Idempotent):** $H^2 = H$ — chiếu hai lần bằng chiếu một lần
* **Đối xứng:** $H^T = H$
* **Vết:** $\text{tr}(H) = \text{rank}(X) = p+1$ (số tham số ước lượng)
* **Leverage** $h_{ii} = [H]_{ii}$: đo mức độ ảnh hưởng của quan sát $i$

**Cài đặt — Economic SVD ($X = U\Sigma V^T$):**
$$H = UU^T$$
Bộ nhớ hiệu quả: chỉ cần $U \in \mathbb{R}^{n \times (p+1)}$ thay vì lưu $H \in \mathbb{R}^{n \times n}$.

In [ ]:
H = hat_matrix(X_design)

# Kiem tra tinh luy dang H^2 = H
idempotent_err = np.max(np.abs(H @ H - H))

# Kiem tra doi xung H^T = H
symmetric_err = np.max(np.abs(H - H.T))

# Vet tr(H) = p + 1
trace_H = np.trace(H)

# Kiem chung H vs numpy
H_ref = X_design @ np.linalg.pinv(X_design)
recon_err = np.max(np.abs(H - H_ref))

print(f'H shape              : {H.shape}')
print(f'Idempotent max|H2-H| : {idempotent_err:.2e}')
print(f'Symmetric  max|H-HT| : {symmetric_err:.2e}')
print(f'tr(H)                : {trace_H:.6f}  (expected {p+1})')
print(f'Max|H - H_numpy|     : {recon_err:.2e}')

# Leverage h_ii
leverage = np.diag(H)
print(f'Leverage h_ii: min={leverage.min():.4f}, mean={leverage.mean():.4f}, max={leverage.max():.4f}')
print(f'sum(h_ii) = tr(H) = {leverage.sum():.4f}')

* **Nhận xét:** Tất cả tính chất toán học của Hat matrix được thỏa mãn với sai số $< 10^{-14}$. Cài đặt qua $H = UU^T$ (Economic SVD) cho kết quả đúng và tiết kiệm bộ nhớ hơn tính trực tiếp $(X^TX)^{-1}$.

### 3. Ridge Regression — Hồi quy với Điều chuẩn L2

**Bài toán tối ưu:**
$$\hat{\beta}_{\text{ridge}} = \arg\min_{\beta} \left\{ \| y - X\beta \|^2 + \lambda \| \beta \|^2 \right\}$$

**Nghiệm dạng đóng (thêm $\lambda I$ vào điều kiện cực tiểu):**
$$\hat{\beta}_{\text{ridge}} = (X^T X + \lambda I)^{-1} X^T y$$

**Tính chất:**
* Khi $\lambda = 0$: thu về OLS thông thường.
* Khi $\lambda \to \infty$: $\hat{\beta} \to 0$ (co về gốc tọa độ).
* $(X^TX + \lambda I)$ luôn khả nghịch với $\lambda > 0$ — giải quyết đa cộng tuyến.

**Ridge Trace:** Đồ thị các hệ số $\hat{\beta}_j$ theo $\lambda$ cho thấy hiệu ứng co về 0 khi $\lambda$ tăng.

In [ ]:
# Du lieu co da cong tuyen (multicollinear)
np.random.seed(0)
n_r, p_r = 80, 4
X_base = np.random.randn(n_r, p_r)
# Them cot gan tuyen tinh phu thuoc
X_multi = np.hstack([X_base, X_base[:, [0]] + 0.01 * np.random.randn(n_r, 1)])
y_r = X_multi @ np.array([1.5, -1.0, 2.0, -0.5, 0.8]) + np.random.randn(n_r)

lambdas = np.logspace(-3, 3, 200)
betas_custom = []
betas_sklearn = []

for lam in lambdas:
    betas_custom.append(ridge_fit(X_multi, y_r, lam=lam))
    betas_sklearn.append(
        SklearnRidge(alpha=lam, fit_intercept=False).fit(X_multi, y_r).coef_
    )

betas_custom = np.array(betas_custom)
betas_sklearn = np.array(betas_sklearn)

max_diff = np.max(np.abs(betas_custom - betas_sklearn))
print(f'Max |custom - sklearn| tren toan bo lambdas: {max_diff:.2e}')

# Ve Ridge Trace
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for j in range(betas_custom.shape[1]):
    axes[0].plot(np.log10(lambdas), betas_custom[:, j], label=fr'$\beta_{j}$')
axes[0].axhline(0, color='k', linestyle='--', linewidth=0.8)
axes[0].set_xlabel(r'$\log_{10}(\lambda)$')
axes[0].set_ylabel('He so')
axes[0].set_title('Ridge Trace (custom)')
axes[0].legend()

for j in range(betas_sklearn.shape[1]):
    axes[1].plot(np.log10(lambdas), betas_sklearn[:, j], label=fr'$\beta_{j}$')
axes[1].axhline(0, color='k', linestyle='--', linewidth=0.8)
axes[1].set_xlabel(r'$\log_{10}(\lambda)$')
axes[1].set_title('Ridge Trace (sklearn - tham khao)')
axes[1].legend()

plt.tight_layout()
plt.show()

* **Nhận xét:** Hai Ridge Trace (custom vs sklearn) hoàn toàn khớp nhau (sai số $< 10^{-12}$). Khi $\lambda$ tăng, các hệ số co dần về 0 — đúng với lý thuyết điều chuẩn Tikhonov.

### 4. VIF — Nhân tử Phóng đại Phương sai

VIF đo mức độ **đa cộng tuyến** (multicollinearity) của từng biến độc lập:
$$VIF_j = \frac{1}{1 - R^2_j}$$

Trong đó $R^2_j$ là hệ số xác định khi hồi quy $x_j$ lên tất cả các biến còn lại.

**Quy tắc chẩn đoán:**
* $VIF_j \approx 1$: Không có đa cộng tuyến
* $5 < VIF_j < 10$: Đa cộng tuyến vừa phải, cần chú ý
* $VIF_j > 10$: Đa cộng tuyến nghiêm trọng, cần xử lý (dùng Ridge hoặc loại biến)

**Cài đặt:** Với mỗi cột $j$, gọi `ols_fit` (qua SVD) để tính $R^2_j$.

In [ ]:
# Kich ban 1: Du lieu doc lap (VIF ~ 1)
np.random.seed(42)
n_v, p_v = 100, 4
X_indep = np.random.randn(n_v, p_v)
vif_indep = vif(X_indep)
print('VIF - Du lieu doc lap:')
print(vif_indep.to_string(index=False))

print()

# Kich ban 2: Da cong tuyen cao (VIF >> 10)
X_collinear = X_indep.copy()
# Cot 3 = 3 * cot 0 + cot 1 + nhieu nho
X_collinear[:, 3] = 3.0 * X_indep[:, 0] + X_indep[:, 1] + 0.05 * np.random.randn(n_v)
vif_collinear = vif(X_collinear)
print('VIF - Du lieu da cong tuyen (cot 3 ~ 3*cot0 + cot1):')
print(vif_collinear.to_string(index=False))

# Kiem chung voi statsmodels
from statsmodels.stats.outliers_influence import variance_inflation_factor
import pandas as pd
ref_vals = [variance_inflation_factor(X_collinear, j) for j in range(p_v)]
print()
print('VIF statsmodels (kiem chung):', [round(v,4) for v in ref_vals])
print('VIF custom      :', [round(v,4) for v in vif_collinear['VIF_Score'].tolist()])
print('Max diff:', max(abs(a-b) for a,b in zip(ref_vals, vif_collinear['VIF_Score'])))

* **Nhận xét:**
  * **Kịch bản 1** (độc lập): VIF ≈ 1 → không có đa cộng tuyến, đúng như thiết kế.
  * **Kịch bản 2** (cột 3 ≈ 3·cột0 + cột1): VIF cột 0, 1, 3 rất lớn (>>10) → phát hiện đúng đa cộng tuyến nghiêm trọng.
  * Kết quả của `vif()` tự cài đặt khớp với `statsmodels.variance_inflation_factor`, xác nhận tính đúng đắn của cài đặt.